# 03 · Partial-occupancy substitution result

**When to use this notebook**

Your target composition has fractional stoichiometry (e.g., `BaFe0.5Mn0.5O3`, `LiFe0.5Mn0.5PO4`, `Na0.5K0.5Cl`), so the substitution engine may produce a structure where a single crystallographic site is shared by two elements with fractional occupancies. `Structure.is_ordered` returns `False` for such a structure, and MatterSim cannot relax it directly.

**What this notebook shows**

The behaviour of `predict_from_formula` in that case: substitution succeeds, the substituted CIF is written to disk, MatterSim relaxation is skipped automatically, and the returned `PredictionResult` has `status='PARTIAL_OCCUPANCY'` with warnings recorded.

**Demo target**: **BaFe0.5Mn0.5O3** with known SG = **221 (Pm-3m cubic perovskite)** — a mixed B-site perovskite. Fe and Mn are both transition metals, so they occupy the same B-site with 0.5/0.5 occupancy.

In [1]:
from pathlib import Path
from csp_workflow_mp import predict_from_formula

# Data paths — set to None to use package defaults from csp_workflow_mp/_paths.py
# (which respect the CSP_MP_DATA_ROOT env var). Override with your own paths if needed.
CIF_DIR = None      # e.g. Path('D:/csp_mp_data/cifs')
META    = None      # e.g. Path('data/MP/metadata_with_descriptors.csv')
OUT     = Path('predictions_demo/partial_occ')     # relative to your cwd

## Run the prediction

`do_relax=True` — relaxation is attempted, but the pipeline detects `is_ordered=False` after substitution and skips MatterSim automatically. The substituted CIF is still written to `output_dir`.

In [2]:
result = predict_from_formula(
    'BaFe0.5Mn0.5O3',
    known_sg=221,
    n_retry=20,
    metadata_csv=META,
    cif_dir=CIF_DIR,
    output_dir=OUT,
    do_relax=True,
    verbose=True,
)

  ✓ substituted CIF saved: predictions_demo\partial_occ\BaFe0.5Mn0.5O3_substituted.cif
  ! partial occupancy detected — MatterSim relaxation skipped
    - Substituted structure contains partial site occupancies (is_ordered=False).
    - MatterSim cannot relax disordered structures directly; the substituted CIF is on disk but has not been relaxed.


C:\ProgramData\miniforge3\envs\csp\lib\_collections_abc.py:824: DeprecationWarning: dict interface is deprecated. Use attribute interface instead
  return self[key]
C:\ProgramData\miniforge3\envs\csp\lib\site-packages\pymatgen\core\structure.py:2945: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Sr0', 'Tc1', 'Tc1', 'O2', 'O3', 'O4']`.
  writer: Any = CifWriter(self, **kwargs)


## Inspect the result

`result` is a `PredictionResult` dataclass. The fields below show the same information as the verbose log above but accessed programmatically — useful for automated downstream processing (e.g., a batch script iterating over many targets and branching on `result.status`).

In [3]:
print(f'status:                {result.status}')
print(f'is_ordered:            {result.is_ordered}')
print(f'substituted CIF path:  {result.substituted_cif_path}')
print(f'relaxed_structure:     {result.relaxed_structure}')
print()
print('warnings recorded by predict_from_formula:')
for w in result.warnings:
    print(f'  - {w}')

status:                PARTIAL_OCCUPANCY
is_ordered:            False
substituted CIF path:  predictions_demo\partial_occ\BaFe0.5Mn0.5O3_substituted.cif
relaxed_structure:     None

warnings recorded by predict_from_formula:
  - Substituted structure contains partial site occupancies (is_ordered=False).
  - MatterSim cannot relax disordered structures directly; the substituted CIF is on disk but has not been relaxed.


## Peek at the partial-occupancy sites

Confirm that the substituted structure has one or more sites shared by multiple species.

In [4]:
s = result.substituted_structure
if s is not None:
    print(f'Substituted structure: {len(s)} sites, is_ordered={s.is_ordered}')
    for i, site in enumerate(s):
        occ_str = ', '.join(f'{sp.symbol}:{occ:.3f}' for sp, occ in site.species.items())
        marker = '  <-- partial occupancy' if not site.is_ordered else ''
        print(f'  site {i}: {occ_str}{marker}')

Substituted structure: 5 sites, is_ordered=False
  site 0: Ba:1.000
  site 1: Fe:0.500, Mn:0.500  <-- partial occupancy
  site 2: O:1.000
  site 3: O:1.000
  site 4: O:1.000


## What the pipeline does with this result

The substituted CIF is on disk. Inside the LOEO benchmark (`scripts/05_run_benchmark.py`), the same case would be recorded as `sub_success=True, relax_converged=False` in `benchmark_raw.csv` and excluded from the paper's valid subset used for SG-match, SOAP, and RMSD statistics.

The pipeline stops here for partial-occupancy structures. Rigorous treatment of configurational disorder — e.g., applying a dominant-species approximation before relaxation, generating multiple ordered configurations, using special quasirandom structures, or combining the workflow with enumeration and cluster-expansion methods — is outside the scope of this repository; the accompanying paper discusses these as future directions.